<a href="https://colab.research.google.com/github/Akhil-kayyamReddy/Internship-Training-AI-driven-project-/blob/main/fraud_detection1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

2. Load Data

In [2]:
df = pd.read_csv("/Fraud.csv")

print(df.shape)
df.head()

(274057, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0.0,0.0


3. Data Understanding

In [3]:
df.info()
df.describe()

# Missing values
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274057 entries, 0 to 274056
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            274057 non-null  int64  
 1   type            274057 non-null  object 
 2   amount          274057 non-null  float64
 3   nameOrig        274056 non-null  object 
 4   oldbalanceOrg   274056 non-null  float64
 5   newbalanceOrig  274056 non-null  float64
 6   nameDest        274056 non-null  object 
 7   oldbalanceDest  274056 non-null  float64
 8   newbalanceDest  274056 non-null  float64
 9   isFraud         274056 non-null  float64
 10  isFlaggedFraud  274056 non-null  float64
dtypes: float64(7), int64(1), object(3)
memory usage: 23.0+ MB


,0
step,0
type,0
amount,0
nameOrig,1
oldbalanceOrg,1
newbalanceOrig,1
nameDest,1
oldbalanceDest,1
newbalanceDest,1
isFraud,1


4. Feature Engineering (30+ FEATURES)

In [4]:
# Balance differences
df['balanceDiffOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balanceDiffDest'] = df['newbalanceDest'] - df['oldbalanceDest']

# Ratios
df['amountToBalanceRatio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
df['destBalanceRatio'] = df['newbalanceDest'] / (df['oldbalanceDest'] + 1)

# Flags
df['isBalanceZeroOrig'] = (df['oldbalanceOrg'] == 0).astype(int)
df['isBalanceZeroDest'] = (df['oldbalanceDest'] == 0).astype(int)

# High transaction
df['isHighAmount'] = (df['amount'] > df['amount'].mean()).astype(int)

# Frequency
df['senderTxnCount'] = df.groupby('nameOrig')['amount'].transform('count')
df['receiverTxnCount'] = df.groupby('nameDest')['amount'].transform('count')

# Avg amount
df['senderAvgAmount'] = df.groupby('nameOrig')['amount'].transform('mean')
df['receiverAvgAmount'] = df.groupby('nameDest')['amount'].transform('mean')

# Deviations
df['senderDeviation'] = df['amount'] - df['senderAvgAmount']
df['receiverDeviation'] = df['amount'] - df['receiverAvgAmount']

# Behavioral flags
df['isNewSender'] = (df['senderTxnCount'] == 1).astype(int)
df['isNewReceiver'] = (df['receiverTxnCount'] == 1).astype(int)

# Fraud patterns
df['suddenDrop'] = (df['balanceDiffOrig'] > df['amount'] * 0.9).astype(int)
df['largeTxn'] = (df['amount'] > 200000).astype(int)

# Encode type
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# Drop IDs
df.drop(['nameOrig', 'nameDest'], axis=1, inplace=True)

print("Total Features:", df.shape[1])

Total Features: 29


5. Data Preparation

In [8]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

6. Handle Imbalance

In [9]:
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

7. Model Training

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

8. Evaluation

In [20]:
y_pred = model.predict(X_test)

non_nan_indices = ~y_test.isna()
y_test_filtered = y_test[non_nan_indices]
y_pred_filtered = y_pred[non_nan_indices]

print(classification_report(y_test_filtered, y_pred_filtered))
print("ROC-AUC:", roc_auc_score(y_test_filtered, y_pred_filtered))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     54779
         1.0       0.93      0.81      0.87        32

    accuracy                           1.00     54811
   macro avg       0.96      0.91      0.93     54811
weighted avg       1.00      1.00      1.00     54811

ROC-AUC: 0.9062317448292229


9. Feature Importance

In [21]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

,0
suddenDrop,0.265702
amountToBalanceRatio,0.199050
balanceDiffOrig,0.109652
newbalanceOrig,0.102563
step,0.047928
receiverAvgAmount,0.032667
type_PAYMENT,0.029333
type_TRANSFER,0.028915
isNewReceiver,0.026189
receiverTxnCount,0.022505
